# **Unbursted**
## A threshold model to cut off burst events in FCS measurements and estimate free molecule populations
---
### This project is organized in 3 parts:

1. **Simulation engine**: build from scratch a simulation generating timestamps comparable to real FCS data
2. **Free molecules validation**: simulate realistic free-diffusion-only conditions and compare against real experimental data, estimating molecule number via **autocorrelation functions** and **3D-diffusion curve fitting**
3. **Free/bound separation**: simulate realistic mixed populations (e.g. DNA-bound and free EtBr molecules), develop a method to remove burst events, then estimate free molecule number via autocorrelation and **3D-diffusion fitting**

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
import time
import pycorrelate as pyc
from scipy.optimize import curve_fit
# load integrated toolkit functions in the project repo
import toolkit as tk
# define paths to handle data smoothly
path_simts = './Pybromo_timestamps/'
path_realts_m1 = './Real_data/m1/'
path_realts_m2 = './Real_data/m2/'

In [ ]:
def diffusion_3d_single(tau, tau_diff, A0):
    # single population, 3D model, kappa=4
    W0 = 0.25e-6
    Z0 = 1.0e-6
    KAPPA = Z0 / W0
    return 1 + A0 / (1 + tau/tau_diff) / np.sqrt(1 + tau/(KAPPA**2 * tau_diff))

def diffusion_3d_double(tau, tau_diff_1, A0_1, tau_diff_2, A0_2):
    # two superposed populations, 3D model, kappa=4
    W0 = 0.25e-6
    Z0 = 1.0e-6
    KAPPA = Z0 / W0
    term1 = A0_1 / (1 + tau/tau_diff_1) / np.sqrt(1 + tau/(KAPPA**2 * tau_diff_1))
    term2 = A0_2 / (1 + tau/tau_diff_2) / np.sqrt(1 + tau/(KAPPA**2 * tau_diff_2))
    return 1 + term1 + term2

def diffusion_3d_single_offset(tau, tau_diff, A0, C):
    # single population and offset, 3D model, kappa=4
    W0 = 0.25e-6
    Z0 = 1.0e-6
    KAPPA = Z0 / W0
    term = A0 / (1 + tau/tau_diff) / np.sqrt(1 + tau/(KAPPA**2 * tau_diff))
    return C + term

def fit_G(tau, G, guesses, bounds, model="single", tau_min=0, label="", sigma=None):
    mask = tau > tau_min
    sigma_m = sigma[mask] if sigma is not None else None
    if model == "single":
        func = diffusion_3d_single
        popt, pcov = curve_fit(func, tau[mask], G[mask], p0=guesses, bounds=bounds, sigma=sigma_m, absolute_sigma=False)
        print(f"[{label}] tau_diff = {popt[0]:.4g} s   A0 = {popt[1]:.4g}")
    elif model == "single_offset":
        func = diffusion_3d_single_offset
        popt, pcov = curve_fit(func, tau[mask], G[mask], p0=guesses, bounds=bounds, sigma=sigma_m, absolute_sigma=False)
        print(f"[{label}] tau_diff = {popt[0]:.4g} s   A0 = {popt[1]:.4g}   C = {popt[2]:.4g}")
    elif model == "double":
        func = diffusion_3d_double
        popt, pcov = curve_fit(func, tau[mask], G[mask], p0=guesses, bounds=bounds, sigma=sigma_m, absolute_sigma=False)
        print(f"[{label}] tau1 = {popt[0]:.4g} s  A0_1 = {popt[1]:.4g}"
              f"   tau2 = {popt[2]:.4g} s  A0_2 = {popt[3]:.4g}")
    else:
        raise ValueError(f'model "{model}" non valida: usa "single", "single_offset" o "double"')
    return popt, pcov

def estimate_background(ts, T_total, dt_bin=2e-3, pct=2):
    nb = int(T_total / dt_bin)
    counts = np.histogram(ts, bins=np.linspace(0, T_total, nb+1))[0]
    return np.percentile(counts, pct) / dt_bin   # Hz

# I. **free EtBr** comparison (hash: dda833)

#### In this section, there will be presented a full analysis of both simulated and real data, based **only** on free EtBr solution as the reference model

In [ ]:
ts_01_apd_1 = tk.load_asc_timestamps(path_realts_m1, 'Sol01_m1.asc')
# I cut because I want to see only the first second from starting run (a sample)
ts_01_apd_1_cut = ts_01_apd_1[ts_01_apd_1 < 60]
# load simulated data
ts_01_sim = tk.load_sim_timestamps(path_simts, 'times_dda833_P954_D4.2e-10_6336pM_step5.0us_t_max60.0s_ID0-0.hdf5')

#### Compare histograms "by-eye"

In [ ]:
bins = np.arange(0,60,1e-2)
stats = tk.trace_stats(ts_01_sim, bins = bins)
thr = stats['median'] - stats['MAD'] 

fig = plt.figure(figsize=(15,5))
ax = fig.add_subplot(3,1,1)
ax.set_title("Real data experiment: free EtBr")
ax.hist(ts_01_apd_1_cut, bins=bins, color="tab:blue", lw=0.01, label='APD 1')
ax.hlines(thr, 0, 60, 'red')
ax.set_xlabel('time [s]')
ax.set_ylabel('Counts')
ax.legend()
ax = fig.add_subplot(3,1,2)
ax.hist(ts_01_sim, bins=bins, color="tab:orange", lw=1, label='Simulated data')
ax.hlines(thr, 0, 60, 'red')
ax.set_xlabel('time [s]')
ax.set_ylabel('Counts')
ax.legend()
ax = fig.add_subplot(3,1,3)
ax.hist(ts_01_apd_1_cut, bins=bins, color="blue", lw=1, label='APD 1')
ax.hist(ts_01_sim, bins=bins, color="red", lw=1, alpha=0.6, label='Simulated data')
ax.set_xlabel('time [s]')
ax.set_ylabel('Counts')
ax.legend()
plt.draw()

#### Estimate average count rate

In [ ]:
Average_Count_Rate_APD_1 =  ts_01_apd_1_cut.size / ts_01_apd_1_cut.max()
Average_Count_Rate_APD_sim = ts_01_sim.size / ts_01_sim.max() 
print(f"ACR APD 1 [RE]: {Average_Count_Rate_APD_1:.2f}")
print(f"ACR [SM]: {Average_Count_Rate_APD_sim:.2f}")

#### Compute autocorrelation function, fit with 3d-diffusion model

In [ ]:
# PSF of confocal volume
W0 = 0.25e-6
Z0 = 1.0e-6
KAPPA = Z0 / W0  # structure parameter = 4.0


# Fit and analysis: two ACFs for real data (APD_1 and APD_2) + ACF from simulated data
unit_re, unit_sm = np.diff(ts_01_apd_1).min(), 5e-6   # native time resolutions
unit_re = unit_re * 1.0
T_RE, T_SM = ts_01_apd_1.max(), ts_01_sim.max()

tau1, G1, Y1 = tk.compute_G(ts_01_apd_1, ts_01_apd_1, unit_re, T_RE)
tau_sm, G_sm, Y_sm = tk.compute_G(ts_01_sim, ts_01_sim, unit_sm, T_SM)

# initial guesses and bounds (reasonable general parameters)
pars=[4e-5, 1.0]
bounds=([1e-7, 1e-3], [1e-2, 1e3])

popt1, pcov1 = fit_G(tau1, G1, pars, bounds, tau_min=5e-6, label="ACF det1")
popt_sm, pcov_sm = fit_G(tau_sm, G_sm, pars, bounds, label="Simulated")

# poptX[0] is the tau_diff from ACFs fit results
# Now the estimated diffusion coefficient will be computed 
D1 = W0**2 / (4 * popt1[0])
D_sm = W0**2 / (4 * popt_sm[0])

print(f"D (det1)     = {D1*1e4:.4g} cm^2/s")   # *1e4 to convert from m^2/s to cm^2/s
print(f"D (sim) = {D_sm*1e4:.4g} cm^2/s")

#### Plot analysis and compare results

In [ ]:
fig = plt.figure(figsize=(10,10))
ax = fig.add_subplot(1,1,1)
ax.plot(tau1, G1, color='tab:blue', ms=3, label='Raw ACF det1', lw=0.5)
ax.plot(tau_sm, G_sm, color='tab:red', ms=3, label='Raw simulated', lw=0.5)
ax.plot(tau1, diffusion_3d_single(tau1, *popt1), 'b-', label='Fit det1')
ax.plot(tau_sm, diffusion_3d_single(tau_sm, *popt_sm), 'r-', label='Fit simulated')
ax.set_xlabel("tau [s]")
ax.set_ylabel("G")
ax.set_ylim(0.8, 1.8)
ax.set_xscale('log')
ax.grid(True, which="both", linestyle="--", alpha=0.5)
ax.legend()
plt.draw()

#### Calculate n_avg_molecules from fit and compare results

In [ ]:
B_sm = 275.0          # bg_rate from simulation
F_sm = Average_Count_Rate_APD_sim
A0_sm = popt_sm[1]

N_sm = (1 - B_sm/F_sm)**2 / A0_sm
print(f"N_eff (simulated, from fit) = {N_sm:.3g}")

# 954 are the N total molecules generated in simulation
N_eff_expected = 954 * (3.48e-16 / 2.5e-13)
print(f"N_eff expected in N_box: {N_eff_expected:.3g}")

# implicit estimation; because simulation because sim is "adapted" to realistic experiment,
# I assume similar value for real experiment
B_re1 = 275.0  
F_re1 = ts_01_apd_1.size / ts_01_apd_1.max()   # = Average_Count_Rate_APD_1
N_re_det1 = (1 - B_re1/F_re1)**2 / popt1[1] 

print(f"N_eff (det1) = {N_re_det1:.3g}")

# II. **EtBr-DNA** comparison (0.06 'blobs' in conf. vol.) (hash: ac470b)

In [ ]:
ts_03_apd_1 = tk.load_asc_timestamps(path_realts_m1, 'Sol03_m1.asc')
# I cut because I want to see only the first second from starting run (a sample)
ts_03_apd_1_cut = ts_03_apd_1[ts_03_apd_1 < 60]
# load simulated data
ts_03_sim = tk.load_sim_timestamps(path_simts, 'times_ac470b_P954_D4.2e-10_P57_D5e-13_6715pM_step5.0us_t_max60.0s_ID0-0.hdf5')

#### Compare histograms "by-eye"

In [ ]:
granularity = 1e-2
bins = np.arange(0,60,granularity)
stats_03_sim = tk.trace_stats(ts_03_sim, bins = bins)
counts_03_sim = stats_03_sim['hist_bars']
stats_03_re = tk.trace_stats(ts_03_apd_1_cut, bins = bins)
counts_03_re = stats_03_re['hist_bars']


thr_sim = stats_03_sim['median']
thr_re = stats_03_re['median']
MAD_sim = stats_03_sim['MAD']
MAD_re = stats_03_re['MAD']

k_re = k_sim = 1.0
thr_sim = thr_sim + k_sim * MAD_sim
thr_re = thr_re + k_re * MAD_re

ts_03_sim_unbursted = tk.unburst(ts_array=ts_03_sim, bins=bins, threshold=thr_sim)
ts_03_re_unbursted = tk.unburst(ts_array=ts_03_apd_1_cut, bins = bins, threshold= thr_re)


In [ ]:
fig = plt.figure(figsize=(15,8), dpi=150)
ax = fig.add_subplot(3,1,1)
ax.set_title("Real data experiment: bound EtBr (n avg DNA-EtBr blobs = 0.06)")
ax.bar(bins[:-1], counts_03_re, width=granularity, color="tab:orange", label='APD 1')
ax.hlines(thr_re,0, 60, 'red', '--', label = f'median: {thr_re}')
ax.hist(ts_03_re_unbursted, bins = bins, color="tab:brown", label='unbursted data')
ax.set_xlabel('time [s]')
ax.set_ylabel('Counts')
ax.legend()

ax = fig.add_subplot(3,1,2)
ax.bar(bins[:-1], counts_03_sim, width=granularity, color="tab:orange", label='Simulated data', alpha=1)
ax.hlines(thr_sim ,0, 60, 'red', '--', label = f'median: {thr_sim:.2f}')
ax.hist(ts_03_sim_unbursted, bins = bins, color="tab:brown", label='unbursted data')
ax.set_xlabel('time [s]')
ax.set_ylabel('Counts')
ax.legend()

ax = fig.add_subplot(3,1,3)
ax.plot(bins[:-1], counts_03_re, drawstyle='steps-mid', color="tab:blue", label='APD 1')
ax.plot(bins[:-1], counts_03_sim, drawstyle='steps-mid', color="tab:orange", label='Simulated data')
ax.set_xlabel('time [s]')
ax.set_ylabel('Counts')
ax.legend()
plt.draw()

#### Estimate average count rate (legacy, to be removed)

In [ ]:
Average_Count_Rate_APD_1 =  ts_03_apd_1_cut.size / ts_03_apd_1_cut.max()
Average_Count_Rate_APD_sim = ts_03_sim.size / ts_03_sim.max() 
print(f"ACR APD 1 [RE]: {Average_Count_Rate_APD_1:.2f}")
print(f"ACR [SM]: {Average_Count_Rate_APD_sim:.2f}")

#### Compute autocorrelation function, fit with 3d-diffusion model

In [ ]:
# PSF of confocal volume
W0 = 0.25e-6
Z0 = 1.0e-6
KAPPA = Z0 / W0  # structure parameter = 4.0

# ACF of real unbursted data + ACF of simulated unbursted data
unit_sm = 5e-6
T_RE = ts_03_re_unbursted.max()         
T_SM = ts_03_sim_unbursted.max()

tau1, G1, Y1 = tk.compute_G(ts_03_re_unbursted, ts_03_re_unbursted, unit_sm, T_RE)
tau_sm, G_sm, Y_sm = tk.compute_G(ts_03_sim_unbursted, ts_03_sim_unbursted, unit_sm, T_SM)

# calculate weights: more pairs in the bins, more reliable estimation
sigma1   = 1.0 / np.sqrt(np.maximum(Y1, 1))
sigma_sm = 1.0 / np.sqrt(np.maximum(Y_sm, 1))

# double-population model: (tau_free, A0_free, tau_bound, A0_bound)
# tau_free fixed to 36.5 us (docs/2): lower==upper 
pars   = [36.5e-6, 0.10, 15e-3, 0.20]
bounds = ([36.49e-6, 1e-3, 1e-3, 1e-4], [36.51e-6, 3.0, 1e-1, 3.0])

popt1, pcov1 = fit_G(tau1, G1, pars, bounds, model="double", tau_min=5e-6,
                     label="ACF det1", sigma=sigma1)
popt_sm, pcov_sm = fit_G(tau_sm, G_sm, pars, bounds, model="double",
                         label="Simulated", sigma=sigma_sm)


#### Plot analysis and compare results

In [ ]:
fig = plt.figure(figsize=(10,7))
ax = fig.add_subplot(1,1,1)
ax.set_title("Autocorrelation Functions - Comparison (case 0.06 'blobs' DNA)")
ax.plot(tau1, G1, color='tab:blue', ms=3, label='Raw ACF det1', lw=0.5)
ax.plot(tau_sm, G_sm, color='tab:red', ms=3, label='Raw simulated', lw=0.5)
ax.plot(tau1, diffusion_3d_double(tau1, *popt1), 'b-', label='Fit det1 (2pop)')
ax.plot(tau_sm, diffusion_3d_double(tau_sm, *popt_sm), 'r-', label='Fit simulated (2pop)')
ax.set_xlabel("tau [s]")
ax.set_ylabel("G")
#ax.set_ylim(0.8, 1.8)
ax.set_xscale('log')
ax.grid(True, which="both", linestyle="--", alpha=0.5)
ax.legend()
plt.draw()

#### Calculate n_avg_molecules from fit and compare results

In [ ]:
# validation "test": A0_free expected (from generation) vs calculated (from fit)
# criteria here is not "sim is similar to real", but "estimated sim value to compare with configured value"

# fixed N from free etbr population, from simulation parameters
N_free_true = 954 * (3.48e-16 / 2.5e-13)      # from n_avg_free=1.0


F_sm_tot = ts_03_sim.size / ts_03_sim.max()

# true n_avg_bound, considering generated data and fraction with free data
N_bound_true = 954 * (3.48e-16 / 2.5e-13) * (0.06/1.0)

# fixed value from simulation as well
B_sm = 275.0
A0_free_expected = (1 - B_sm/F_sm_tot)**2 / N_free_true

print(f"N_free  (pre-configured from sim)   = {N_free_true:.4g}")
print(f"N_bound (pre-configured from sim)   = {N_bound_true:.4g}")
print(f"A0_free expected   (from simulation) = {A0_free_expected:.4f}")
print(f"A0_free recovered (from fit double pop)      = {popt_sm[1]:.4f}")

print()
print("fraction recovered/expected (near to 1 = fine value):")
print(f"  double : {popt_sm[1] / A0_free_expected:.3f}")

# II. **EtBr-DNA** comparison (0.08 'blobs' in conf. vol.) (hash: 00512f)

In [ ]:
ts_05_apd_1 = tk.load_asc_timestamps(path_realts_m1, 'Sol05_m1.asc')
# I cut because I want to see only the first second from starting run (a sample)
ts_05_apd_1_cut = ts_05_apd_1[ts_05_apd_1 < 60]
# load simulated data
ts_05_sim = tk.load_sim_timestamps(path_simts, 'times_00512f_P954_D4.2e-10_P76_D5e-13_6841pM_step5.0us_t_max60.0s_ID0-0.hdf5')

#### Compare histograms "by-eye"

In [ ]:
granularity = 1e-2
bins = np.arange(0,60,granularity)
stats_05_sim = tk.trace_stats(ts_05_sim, bins = bins)
counts_05_sim = stats_05_sim['hist_bars']
stats_05_re = tk.trace_stats(ts_05_apd_1_cut, bins = bins)
counts_05_re = stats_05_re['hist_bars']


thr_sim = stats_05_sim['median']
thr_re = stats_05_re['median']
MAD_sim = stats_05_sim['MAD']
MAD_re = stats_05_re['MAD']

k_re = 1
k_sim = 1

thr_sim = thr_sim + k_sim * MAD_sim
thr_re = thr_re + k_re * MAD_re

ts_05_sim_unbursted = tk.unburst(ts_array=ts_05_sim, bins=bins, threshold=thr_sim)
ts_05_re_unbursted = tk.unburst(ts_array=ts_05_apd_1_cut, bins = bins, threshold= thr_re)


In [ ]:
fig = plt.figure(figsize=(15,8), dpi=150)
ax = fig.add_subplot(3,1,1)
ax.set_title("Real data experiment: bound EtBr (n avg DNA-EtBr blobs = 0.08)")
ax.bar(bins[:-1], counts_05_re, width=granularity, color="tab:orange", label='APD 1')
ax.hlines(thr_re,0, 60, 'red', '--', label = f'median: {thr_re}')
ax.hist(ts_05_re_unbursted, bins = bins, color="tab:brown", label='unbursted data')
ax.set_xlabel('time [s]')
ax.set_ylabel('Counts')
ax.legend()

ax = fig.add_subplot(3,1,2)
ax.bar(bins[:-1], counts_05_sim, width=granularity, color="tab:orange", label='Simulated data', alpha=1)
ax.hlines(thr_sim ,0, 60, 'red', '--', label = f'median: {thr_sim:.2f}')
ax.hist(ts_05_sim_unbursted, bins = bins, color="tab:brown", label='unbursted data')
ax.set_xlabel('time [s]')
ax.set_ylabel('Counts')
ax.legend()

ax = fig.add_subplot(3,1,3)
ax.plot(bins[:-1], counts_05_re, drawstyle='steps-mid', color="tab:blue", label='APD 1')
ax.plot(bins[:-1], counts_05_sim, drawstyle='steps-mid', color="tab:orange", label='Simulated data')
ax.set_xlabel('time [s]')
ax.set_ylabel('Counts')
ax.legend()
plt.draw()

#### Estimate average count rate (legacy, to be removed)

In [ ]:
Average_Count_Rate_APD_1 =  ts_05_apd_1_cut.size / ts_05_apd_1_cut.max()
Average_Count_Rate_APD_sim = ts_05_sim.size / ts_05_sim.max() 
print(f"ACR APD 1 [RE]: {Average_Count_Rate_APD_1:.2f}")
print(f"ACR [SM]: {Average_Count_Rate_APD_sim:.2f}")

#### Compute autocorrelation function, fit with 3d-diffusion model

In [ ]:
# PSF of confocal volume
W0 = 0.25e-6
Z0 = 1.0e-6
KAPPA = Z0 / W0  # structure parameter = 4.0

# ACF of real unbursted data + ACF of simulated unbursted data
unit_sm = 5e-6
T_RE = ts_05_re_unbursted.max()
T_SM = ts_05_sim_unbursted.max()

tau1, G1, Y1 = tk.compute_G(ts_05_re_unbursted, ts_05_re_unbursted, unit_sm, T_RE)
tau_sm, G_sm, Y_sm = tk.compute_G(ts_05_sim_unbursted, ts_05_sim_unbursted, unit_sm, T_SM)


# calculate weights: more pairs in the bins, more reliable estimation
sigma1   = 1.0 / np.sqrt(np.maximum(Y1, 1))
sigma_sm = 1.0 / np.sqrt(np.maximum(Y_sm, 1))

# double-population model: (tau_free, A0_free, tau_bound, A0_bound)
# tau_free fixed to 36.5 us (docs/2): lower==upper 
pars   = [36.5e-6, 0.10, 15e-3, 0.20]
bounds = ([36.49e-6, 1e-3, 1e-3, 1e-4], [36.51e-6, 3.0, 1e-1, 3.0])

popt1, pcov1 = fit_G(tau1, G1, pars, bounds, model="double", tau_min=5e-6,
                     label="ACF det1", sigma=sigma1)
popt_sm, pcov_sm = fit_G(tau_sm, G_sm, pars, bounds, model="double",
                         label="Simulated", sigma=sigma_sm)

#### Plot analysis and compare results

In [ ]:
fig = plt.figure(figsize=(10,7))
ax = fig.add_subplot(1,1,1)
ax.set_title("Autocorrelation Functions - Comparison (case 0.08 'blobs DNA)")
ax.plot(tau1, G1, color='tab:blue', ms=3, label='Raw ACF det1', lw=0.5)
ax.plot(tau_sm, G_sm, color='tab:red', ms=3, label='Raw simulated', lw=0.5)
ax.plot(tau1, diffusion_3d_double(tau1, *popt1), 'b-', label='Fit det1 (2pop)')
ax.plot(tau_sm, diffusion_3d_double(tau_sm, *popt_sm), 'r-', label='Fit simulated (2pop)')
ax.set_xlabel("tau [s]")
ax.set_ylabel("G")
ax.set_xscale('log')
ax.grid(True, which="both", linestyle="--", alpha=0.5)
ax.legend()
plt.draw()

#### Calculate n_avg_molecules from fit and compare results

In [ ]:
# validation "test": A0_free expected (from generation) vs calculated (from fit)
# criteria here is not "sim is similar to real", but "estimated sim value to compare with configured value"

# fixed N from free etbr population, from simulation parameters
N_free_true = 954 * (3.48e-16 / 2.5e-13)      # from n_avg_free=1.0


F_sm_tot = ts_03_sim.size / ts_03_sim.max()

# true n_avg_bound, considering generated data and fraction with free data
N_bound_true = 954 * (3.48e-16 / 2.5e-13) * (0.06/1.0)

# fixed value from simulation as well
B_sm = 275.0
A0_free_expected = (1 - B_sm/F_sm_tot)**2 / N_free_true

print(f"N_free  (pre-configured from sim)   = {N_free_true:.4g}")
print(f"N_bound (pre-configured from sim)   = {N_bound_true:.4g}")
print(f"A0_free expected   (from simulation) = {A0_free_expected:.4f}")
print(f"A0_free recovered (from fit double pop)      = {popt_sm[1]:.4f}")

print()
print("fraction recovered/expected (near to 1 = fine value):")
print(f"  double : {popt_sm[1] / A0_free_expected:.3f}")